# <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Seminar 2b: Vizualizarea datelor</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/02-explorare-vizualizare/seminar-02b-vizualizarea-datelor.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "02-explorare-vizualizare"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

## Problema de business

Managerul unui magazin online: *„Ce se vinde, cui și când? Și pot avea încredere în cifrele din sistem?”*

- **Rolul vostru:** analist de date: explorați datele, găsiți problemele de calitate și arătați tiparele importante.
- **Ce predați la final:** un raport de o pagină pe tranzacțiile din `lab-set4`: trei grafice, fiecare cu un titlu care spune concluzia, plus lista problemelor găsite în date.

## Cum lucrăm azi (2 ore)

| Timp | Activitate |
|---|---|
| 10 min | Problema de business, graficul potrivit întrebării, cum se citește un boxplot (mai jos) |
| 40 min | **Exemple ghidate** 1, 2, 3 și 6 (linie, bare, histogramă, boxplot). Exemplele 4, 5, 7 și bonusul rămân ca referință pentru acasă |
| 60 min | **Exercițiile 1-3 pe `lab-set4`**: voi scrieți cererile către asistentul AI, rulați, verificați cu **valorile de control**, interpretați |
| 10 min | **Discuție:** ce a greșit asistentul și cum v-ați dat seama |

La fiecare exercițiu: scrieți cererea **în cuvintele voastre** (vezi rețeta din [ghidul AI](https://github.com/sbildea/SEED/blob/main/docs/ghid-programare-asistata-ai.md)), lipiți codul generat, rulați, comparați cu valorile de control, bifați verificările și răspundeți la întrebarea de interpretare.
Operațiile cu dataframe-uri de care aveți nevoie (filtrare, grupare, coloane noi) sunt în [seminar-02a-dataframe](seminar-02a-dataframe.ipynb): folosiți-le ca vocabular în cereri.

## Graficul potrivit întrebării

| Întrebarea | Graficul | Exemplul |
|---|---|---|
| Cum evoluează ceva în timp? | linie (line plot) | 1 |
| Cum se compară categoriile între ele? | bare (bar chart) | 2 |
| Cum sunt distribuite valorile unei variabile? | histogramă | 3 |
| Cum se compară distribuțiile și unde sunt valorile aberante? | boxplot | 6 |
| Ce parte din total are fiecare categorie? | pie / donut (doar pentru **sume**, nu pentru medii) | 4, 5 |
| Există o legătură între două variabile numerice? | scatter | 7 |

**Cum citim un boxplot.** Cutia se întinde de la primul cuartil (Q1, 25%) la al treilea (Q3, 75%): lățimea ei este **IQR = Q3 − Q1**. Linia din cutie este mediana. Mustățile ajung până la cea mai îndepărtată valoare aflată la cel mult **1,5 × IQR** de cutie; ce e dincolo apare ca puncte izolate (posibile valori aberante: erori sau cazuri rare, decizia depinde de întrebare).

In [ ]:
# aceeași variabilă văzută ca histogramă (densitatea) și ca boxplot: prețul caselor, setul 6
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from seed.date import incarca, lista_seturi

pret = incarca("lab-set6")[1]["SalePrice"]
q1, mediana, q3 = pret.quantile([0.25, 0.5, 0.75])
iqr = q3 - q1                      # lățimea cutiei
limita_sus = q3 + 1.5 * iqr        # peste această valoare punctele apar izolate
print(q1, mediana, q3, iqr, limita_sus, (pret > limita_sus).sum())

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
sns.histplot(pret, kde=True, ax=ax[0])
sns.boxplot(x=pret, ax=ax[1])
plt.show()

**Valori de control:** Q1 = 129 975, mediana = 163 000, Q3 = 214 000, IQR = 84 025, limita de sus = 340 037,5; **61** de case sunt deasupra ei.

**Interpretare:** De ce mediana (163 000) este mai mică decât media (≈180 900)? Ce arată coada din dreapta a histogramei și cum o vedeți în boxplot?

In [ ]:
import plotly.graph_objects as go
sns.set_theme()  # aspect unitar pentru graficele seaborn

## Citirea datelor

Seturile de date se încarcă cu funcția `incarca` din pachetul cursului, la fel în Colab și pe calculatorul vostru. Nu trebuie descărcat nimic manual și nu trebuie modificată nicio cale:

```python
date = incarca("lab-set4")   # toate tabelele setului 4
df = date[0]                 # primul tabel (sau după nume: date["OnlineRetail"])
```

In [ ]:
# Citirea datelor de laborator: nu trebuie modificată nicio cale.
# incarca("lab-set4") descarcă setul (doar prima dată) și întoarce tabelele lui;
# un tabel se alege după nume sau după poziție: date["OnlineRetail"] sau date[0].
from seed.date import incarca, lista_seturi
lista_seturi("lab")

## Exemplul 1. 
### Line plots

In [ ]:
dataset = incarca("lab-set4")
dataset[0]

La citire am creat ceva generic, o listă de dataframe-uri, în caz că sunt prezente mai multe tabele. Pentru acest set de date avem un singur tabel și îl vom alege din listă. Explorăm coloanele și ne uităm la câteva înregistrări.

In [ ]:
df1 = dataset[0]

In [ ]:
df1.info()

In [ ]:
df1.head(5)

Ar fi interesant să vedem volumul lunar al vânzărilor. Pentru asta vom construi două noi coloane. Ulterior, când vom vorbi despre serii de timp, vom investiga moduri de agregare prin intermediul reeșantionării bazate pe index temporal. Acum însă vom face agregarea manual.

> ItemRevenue

> InvoiceMonth - asta va da un exemplu de formatare a datelor in Python

In [ ]:
df1 = df1.assign( Revenue = df1['Quantity'] * df1['UnitPrice'])

Un mod eficace de a trece din format `datetime` in str:

In [ ]:
from datetime import datetime
df1 = df1.assign(InvoiceMonth = df1['InvoiceDate'].apply(lambda x: str(100*x.year + x.month)[0:4] +
                                                        '-' +str(100*x.year + x.month)[4:] ))

In [ ]:
df1.head()

Să vedem volumul lunar de vânzări

In [ ]:
df_sample = df1.groupby('InvoiceMonth', as_index = False)['Revenue'].agg('sum')
df_sample.head(5)

Ca și cu toate celelalte, vizualizarea depinde de modulul folosit:

In [ ]:
# un grafic cu punctele unite prin linii - 'line chart', lunile pe axa x, venitul pe y
plt.plot( df_sample.InvoiceMonth, 
         df_sample.Revenue, 
         color='green', marker='o', linestyle='solid')
# add a title
plt.title("Venit lunar")
# add a label to the y-axis
plt.ylabel("Venit")
plt.show()

După cum vedem, etichetele lunilor sunt suprascrise. Pentru a evita acest aspect putem roti etichetele, folosind o proprietate a plt.xticks - `rotation=45`:

In [ ]:
plt.plot(df_sample.InvoiceMonth, df_sample.Revenue, color='green', marker='o', linestyle='solid')
# add a title
plt.title("Venit lunar")
# add a label to the y-axis
plt.ylabel("Venit lunar")
plt.xticks(range(df_sample.shape[0]), df_sample.InvoiceMonth, rotation=45)
plt.show()

Acum sa vedem acelasi grafic cu seaborn

In [ ]:
ax = sns.lineplot(x="InvoiceMonth", y="Revenue", data=df_sample, marker='o')
ax.set_title('Venit lunar')
ax.set_ylabel("Venit")
ax.set_xlabel("Luna")
plt.xticks(rotation=45)
plt.show()

Același grafic cu plotly:

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x = df_sample.InvoiceMonth,
    y= df_sample.Revenue,
    mode = 'markers+lines'
    ))

fig.update_layout( 
    title = 'Venit Lunar',
    xaxis_title = 'Luna',
    yaxis_title = 'Venit')
fig.show()

**Valori de control (datele brute):** venitul lunar este maxim în noiembrie 2011 (≈ 1 461 756) și cel mai mic în decembrie 2011 (≈ 433 668).

**Interpretare:** Scade cu adevărat vânzarea în decembrie 2011? Ce ar trebui să verificați în date înainte să trageți concluzia (pistă: `df1.InvoiceDate.max()`)?

# Exemplul 2. 

## Bar Charts

Un grafic de bare este o alegere bună atunci când doriți să arătați cum variază o anumită cantitate într-un un set de elemente discrete.

In [ ]:
date_set1 = incarca("lab-set1")

In [ ]:
df2=date_set1[0]

In [ ]:
df2.head()

Să vedem distribuția clienților în funcție de pachetul de internet:

In [ ]:
sns.set(style="whitegrid")
ax = sns.countplot(x="InternetService", data=df2)
ax.set_title('Număr clienți')

Dacă în loc de numărare(count) dorim să agregăm pe axa y o altă variabilă vom folosi

`seaborn.barplot`

De exemplu să vedem vechimea medie (`tenure`, în luni) a celor care stau sau pleacă în funcție de sex:

In [ ]:
df2.info()

In [ ]:
ax = sns.barplot( x= "Churn", y = "tenure", 
                 hue = "gender", data = df2)

Observăm în graficul anterior un parametru nou, `hue`, în funcție de care se pot face grupări. 

În cazul nostru puteam să calculăm și vechimea medie a celor care pleacă indiferent de sex. 

Distincția prin hue poate face mai relevantă informația. 

Acest parametru se poate folosi în mai toate graficele seaborn.

Mai jos încă un exemplu cu x si hue interschimbate.

In [ ]:
ax = sns.barplot( x= "gender", y = "tenure", hue = "Churn", data = df2)

Același tip de grafic cu plotly. Să notăm că agregările trebuie să le facem manual

In [ ]:
import plotly.graph_objects as go

# nu putem evita scriere de cod, nu există un parametru echivalent cu hue
fig = go.Figure()
df_to_plot = df2.groupby(['gender','Churn'],as_index = False).agg( medie_abonament = ('tenure','mean'))

# pentru fiecare tip de churn adăugăm o bară
for churn in df_to_plot.Churn.unique():
    fig.add_trace(go.Bar(
        x=df_to_plot[df_to_plot.Churn == churn].gender,
        y=df_to_plot[df_to_plot.Churn == churn].medie_abonament,
        name= str(churn),
        #marker_color='indianred'
    ))

# Change the bar mode
fig.update_layout(barmode='group',
                 title = 'Plecări și media de durată a unui abonament după sex')
fig.show()

**Valori de control:** `InternetService`: Fiber optic 3 096, DSL 2 421, No 1 526. Vechimea medie: ≈ 37-38 luni pentru cei care rămân (`Churn` = No) și ≈ 17-19 luni pentru cei care pleacă (`Churn` = Yes), aproape identică pentru cele două genuri.

**Interpretare:** Care variabilă diferențiază clar clienții (gen sau churn)? Ce reprezintă liniile negre de pe bare?

# Exemplul 3

## Histogram

Histogramele sunt o prezentare aproximativă a densității de distribuție a unei variabile continue.

In [ ]:
date_echipa = incarca("lab-set2")

In [ ]:
df3 = date_echipa[1]
df3.dtypes

In [ ]:
import plotly.graph_objects as go

import numpy as np

x0 = df3.MonthlyIncome


fig = go.Figure()
fig.add_trace(go.Histogram(x=x0))
#fig.add_trace(go.Histogram(x=x1))

# Overlay both histograms
fig.update_layout(barmode='overlay')
# Reduce opacity to see both histograms
#fig.update_traces(opacity=0.75)
fig.show()

Doua distributii in acelasi plot:

In [ ]:
import plotly.graph_objects as go

import numpy as np

x0 = df3[df3.Gender == 'Male'].MonthlyIncome

x1 = df3[df3.Gender == 'Female'].MonthlyIncome

fig = go.Figure()
fig.add_trace(go.Histogram(x=x0, name ='Male'))
fig.add_trace(go.Histogram(x=x1, name = 'Female'))

# Overlay both histograms
fig.update_layout(barmode='overlay')
# Reduce opacity to see both histograms
fig.update_traces(opacity=0.75)
fig.update_layout(    title = 'Distribuția venitului - histogramă comparativă')
fig.show()

Grafice alăturate cu plotly

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# definim o matrice de imagini - numărul de linii și coloane decide așezarea în pagină
fig = make_subplots(rows=1, cols=2)

# imaginea din celula (1,1) - poate fi orice grafic, aici e una dintre histograme
fig.add_trace(go.Histogram(x=x0, name ='Male'),
              row=1, col=1)

# imaginea din celula (1,2)
fig.add_trace(go.Histogram(x=x1, name = 'Female'),
              row=1, col=2)

# adăugăm un titlu
fig.update_layout(    title = 'Distribuția venitului - histogramă comparativă')
fig.show()

**Valori de control (`MonthlyIncome`):** Q1 = 29 110, mediana = 49 190, Q3 = 83 800, asimetria (skew) ≈ 1,37. Pe statut marital, mediana este ≈ 52 090 (Divorced), 48 980 (Married), 46 875 (Single).

**Interpretare:** Distribuția este simetrică? Ce măsură de centru alegeți pentru a descrie „venitul tipic” și de ce?

# Exemplul 4

## Pie charts

In [ ]:
date_echipa = incarca("lab-set3")

In [ ]:
df4 = date_echipa[0]

In [ ]:
df4.dtypes

Un alt mod de a prezenta agregări de categorii, la nivel de proporții, este pie chart: feliile de pizza/plăcintă.

Să vedem proporția copiilor pe tipurile de ocupație ale mamei precum și nota finală medie pe aceste grupuri.

Vom face un pie chart pentru proporții; vom colora specific grupurile și vom eticheta cu tipurile de ocupație.

In [ ]:
df_sample = df4.groupby('Mjob')['absences'].agg(['mean','sum']).reset_index()\
    .sort_values(by = 'sum', ascending = False)

In [ ]:
df_sample.dtypes

Să facem un grafic și cu matplotlib:

In [ ]:
import matplotlib.pyplot as plt

# Data to plot
labels = df_sample['Mjob']
n = df_sample.shape[0]
sizes = df_sample['sum'].to_list()
#colors = ['gold', 'yellowgreen', 'lightcoral', 'lightskyblue','red']
# explode = (0.1, 0.2, 0, 0, 0)  # explode 1st slice
explode = [0.0] * n
explode[0] = 0.2
explode[1] = 0.1
# Plot
plt.pie(sizes, explode=tuple(explode), labels=labels, 
        #colors=colors,
        autopct='%1.1f%%', shadow=True, startangle=140)
plt.title("Procent absențe în funcție de ocupația mamei")
plt.axis('equal')
plt.show()

**Valori de control (suma absențelor pe ocupația mamei):** other 870 (38,6%), services 620 (27,5%), teacher 333 (14,8%), at_home 277 (12,3%), health 155 (6,9%).

**Interpretare:** Pie chart are sens aici pentru că însumăm absențele (părți dintr-un total). Ar avea sens un pie chart al **mediilor** pe grupe? De ce?

# Exemplul 5

## Donut charts

Cu ajutorul acestui tip de grafic se pot reprezenta proportii de mai multe tipuri simultan.

In [ ]:
date_echipa = incarca("lab-set5")

In [ ]:
date_echipa[0].info()

In [ ]:
df5 = date_echipa[0]

Să încercăm să vedem proporția de proprietari (Home Ownership =  owners) care iau împrumut pentru a îmbunătăți casa (Purpose = home improvement)

In [ ]:
df5.groupby('Home Ownership')['Loan ID'].agg('count').index

In [ ]:
# Libraries
import matplotlib.pyplot as plt
 
# enumaram proprietarii
group_names = df5.groupby('Home Ownership')['Loan ID'].agg('count').index

# numaram proprietarii
group_size=df5.groupby('Home Ownership')['Loan ID'].agg('count').values

subgroup_names=df5.groupby(['Home Ownership','Purpose'])['Loan ID'].agg('count').index
subgroup_names=[s[1].replace("'","").replace("}","") for s in subgroup_names]
subgroup_size=df5.groupby(['Home Ownership','Purpose'])['Loan ID'].agg('count').values
 
# Creare culori
my_colors=[plt.cm.Blues, plt.cm.Reds, plt.cm.Greens, plt.cm.Purples, plt.cm.Oranges]
my_color_lenghts = date_echipa[0][['Home Ownership','Purpose']].drop_duplicates().groupby('Home Ownership')['Purpose'].count()

# Inel exterior
fig, ax = plt.subplots()
ax.axis('equal')
mypie, _ = ax.pie(group_size, radius=1.3, labels=group_names, colors=[c(0.6) for c in my_colors] )
plt.setp( mypie, width=0.3, edgecolor='white')
 
colors = []
for i in range(len(my_colors)):
    
    for j in range(subgroup_size[i]):
        colors.append(my_colors[i](0.6- j * 0.1))
        

plt.margins(0,0)
 
# show it
plt.show()

Același grafic cu plotly:

Agregăm datele la nivelul celor două dimensiuni - tipul de reședință actuală și scopul împrumutului.

In [ ]:
df5_agg = df5.groupby(['Home Ownership','Purpose'], as_index = False).agg(
    LoanCount =('Loan ID','count'))
df5_agg

In [ ]:
import plotly.express as px

import numpy as np
df = df5_agg
fig = px.sunburst(df, path=['Home Ownership', 'Purpose'], values='LoanCount',
                  color='Purpose', #hover_data=['iso_alpha'],
                  color_continuous_scale='RdBlu',
                 # color_continuous_midpoint=np.average(df['Purpose'], weights=df['LoanCount'])
                 )
fig.show()

Parallel Category Graph

In [ ]:
import plotly.graph_objects as go
import pandas as pd



# Create dimensions
ho_dim = go.parcats.Dimension(
    values=df5['Home Ownership'],
    categoryorder='category ascending', label="Home Ownership"
)

purpose_dim = go.parcats.Dimension(values=df5.Purpose, label="Purpose")

years_dim = go.parcats.Dimension(
    values=df5['Years in current job'], label="Years in current job"
)

# Create parcats trace
color = df5['Months since last delinquent']
colorscale = [[0, 'lightsteelblue'], [1, 'mediumseagreen']]

fig = go.Figure(data = [go.Parcats(dimensions=[ho_dim, purpose_dim, years_dim],
        line={'color': color, 'colorscale': colorscale},
        hoveron='color', hoverinfo='count+probability',
        labelfont={'size': 18, 'family': 'Times'},
        tickfont={'size': 16, 'family': 'Times'},
        arrangement='freeform')])

fig.show()

**Valori de control:** `date_echipa[0]` este `credit_test` (10 353 de rânduri): suma dimensiunilor inelului exterior trebuie să fie 10 353.

**Interpretare:** Ce aflați dintr-un donut cu două inele care nu vedeați într-un singur pie chart?

# Exemplul 6

## Boxplot

Grafic care prezinta distributia unei variabile folosind precentile. Util pentru compararea unor distributii si a evolutiei in timp, sau pentru comparare alaturata a unor variabile continue asemanatoare

In [ ]:
date_echipa = incarca("lab-set6")
df6 = date_echipa[1]

In [ ]:
df6.YrSold.drop_duplicates()

Aș vrea să văd prețul caselor în boxplot. Apoi aș vrea să văd comparate boxploturi pe ani în același grafic ca să văd 
cum s-a modificat distribuția prețurilor în timp.

In [ ]:
ax = sns.boxplot(y="SalePrice",data=df6)

In [ ]:
ax = sns.boxplot(x="YrSold", y="SalePrice", hue="YrSold", data=df6, palette="Set3", legend=False)

Ames este un orășel universitar din statul Iowa. Vara sunt peste 30 Celsius cu umiditate, iar iarna ajunge și la -20 Celsius uneori pentru zile în șir. 

Să vedem cum este afectat prețul caselor de faptul că au sau nu au instalație mixtă de aer condiționat și de încălzire.

In [ ]:
ax = sns.boxplot(x="YrSold", y="SalePrice", hue="CentralAir",data=date_echipa[1], palette="Set3")

**Valori de control:** prețul median este ≈ 98 000 pentru casele fără aer condiționat central (`CentralAir` = N) și ≈ 168 000 pentru cele cu (Y).

**Interpretare:** Ce înseamnă punctele din afara mustăților? Sunt erori sau case scumpe reale? Diferența între N și Y înseamnă că aerul condiționat *ridică* prețul sau că e legat de altceva (vechimea casei)?

# Exemplul 7 

## Scatterplot

In [ ]:
date_echipa = incarca("lab-set7")

In [ ]:
df7 = date_echipa[6]

In [ ]:
ax = sns.scatterplot(x="InscClaimAmtReimbursed", y="DeductibleAmtPaid", data=date_echipa[6].loc[0:1000])

In [ ]:
df7[['InscClaimAmtReimbursed','DeductibleAmtPaid']].describe()

Vedem ca la despagubiri a 75 percentila e 200USD. Câte cereri de despăgubire au depășit 15000USD?

In [ ]:
df7[df7.InscClaimAmtReimbursed>15000].shape

In [ ]:
ax = sns.scatterplot(x="Provider", 
                     y="InscClaimAmtReimbursed", 
                     data=df7[df7.InscClaimAmtReimbursed>15000],
                     size = "InscClaimAmtReimbursed" )

ax.tick_params(axis='x', rotation=45)

**Valori de control:** a 75-a percentilă a despăgubirilor este 200 USD; **21** de cereri depășesc 15 000 USD.

**Interpretare:** Punctele foarte îndepărtate sunt erori de introducere sau cazuri reale de investigat? Ce ați face înainte să le eliminați?

## Bonus: frecvența cuvintelor (`wordcloud`)

Scopul este o ilustrare grafică a furnizorilor (`Provider`) după numărul de cereri de despăgubire. Pachetul `wordcloud` este în `requirements-local.txt` (în Colab este deja instalat).

In [ ]:
from wordcloud import WordCloud

# numaram cazurile pe asigurator
word_freq = df7.groupby('Provider')['ClaimID'].count()
text = " ".join([(k + " ")*v for k,v in word_freq.items()])

# Generare de word cloud image
wordcloud = WordCloud().generate(text)


# Display imagine generata:
# the matplotlib way:
import matplotlib.pyplot as plt
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")

# Exerciții pe `lab-set4`: de la date brute la raport

Lucrați cu tabelul `OnlineRetail` (`incarca("lab-set4")[0]`). Pentru fiecare exercițiu: cerere în cuvintele voastre, cod generat, **valori de control**, interpretare. Valorile de control se verifică pe datele brute, exact în ordinea indicată.

## Exercițiul 1: Raport de calitate a datelor

**Ce cereți:** un raport pe coloane: tip, număr și procent de valori lipsă, număr de valori distincte; numărul de rânduri duplicate; numărul de cantități negative (`Quantity` < 0) și câte dintre ele sunt pe facturi care încep cu `C`; numărul de prețuri `UnitPrice` egale cu 0 și mai mici ca 0.

**Cererea mea (prompt):**

> *scrieți aici cererea trimisă asistentului*

In [ ]:
# codul generat pentru exercițiul 1

**Valori de control**

| Verificare | Rezultat |
|---|---|
| Dimensiune | 541 909 rânduri × 8 coloane |
| Rânduri duplicate | 5 268 |
| `Description` lipsă | 1 454 |
| `CustomerID` lipsă | 135 080 (24,9%) |
| `Quantity` < 0 | 10 624, din care **9 288** pe facturi care încep cu `C` |
| `UnitPrice` = 0 / < 0 | 2 515 / 2 |

- [ ] am verificat fiecare număr din tabel; dacă diferă, am cerut asistentului să explice ce a numărat;
- [ ] știu ce înseamnă o cantitate negativă care **nu** este pe o factură `C` (1 336 de cazuri).

**Interpretare:** Care dintre aceste probleme ar schimba o analiză a vânzărilor și care nu? Ce faceți cu `CustomerID` lipsă dacă întrebarea e „ce se vinde” și ce faceți dacă întrebarea e „cine cumpără”?

*Răspunsul meu:*

## Exercițiul 2: Valori aberante și retururi

**Ce cereți:** boxplot pentru `Quantity` și pentru `UnitPrice` pe datele brute; apoi curățați: eliminați duplicatele, facturile `C`, rândurile cu `Quantity` ≤ 0 sau `UnitPrice` ≤ 0 și codurile de produs care nu încep cu 5 cifre (`StockCode`: `POST`, `DOT`, `M`, `AMAZONFEE` etc. nu sunt produse); refaceți boxplot-urile.

**Cererea mea (prompt):**

> *scrieți aici cererea trimisă asistentului*

In [ ]:
# codul generat pentru exercițiul 2

**Valori de control**

| Verificare | Rezultat |
|---|---|
| `Quantity` (brut): Q1 / mediana / Q3 | 1 / 3 / 10 (limitele boxplot-ului: −12,5 și 23,5) |
| `Quantity` (brut): minim / maxim | −80 995 / 80 995 |
| `UnitPrice` (brut): Q1 / mediana / Q3 | 1,25 / 2,08 / 4,13 (limita de sus: 8,45); minim −11 062,06, maxim 38 970 |
| Coduri care nu sunt produse | 33 de coduri distincte, 2 995 de rânduri |
| După curățare | **522 504** de rânduri (au fost eliminate 19 405) |

- [ ] pe boxplot-ul brut nu se mai vede cutia: scala e dictată de extreme; știu de ce;
- [ ] știu câte rânduri a eliminat fiecare filtru, nu doar totalul.

**Interpretare:** Sunt retururile erori sau informație utilă? Depinde de întrebare: ce întrebare de business ar cere păstrarea lor?

*Răspunsul meu:*

## Exercițiul 3: Raportul de o pagină

Pe datele **curățate** de la exercițiul 2, faceți trei grafice care răspund la „Ce se vinde, cui și când?”. Fiecare grafic are un **titlu care spune concluzia**, nu subiectul (exemplu: *„Vânzările se dublează între august și noiembrie”*, nu *„Venit lunar”*), axe cu etichete și o propoziție sub grafic.

1. **Când?** venitul (`Quantity` × `UnitPrice`) lunar: linie.
2. **Ce?** primele 5 produse după venit: bare.
3. **Cui?** (la alegere) distribuția pe țări sau câți clienți (`CustomerID`) aduc 80% din venit.

La final, scrieți lista problemelor găsite în date (din exercițiile 1 și 2) și ce decizie ați luat pentru fiecare.

**Cererea mea (prompt):**

> *scrieți aici cererea trimisă asistentului*

In [ ]:
# codul generat pentru exercițiul 3

**Valori de control (date curățate)**

| Verificare | Rezultat |
|---|---|
| Venit total | ≈ 10 246 821 |
| Cel mai bun produs | 22423 REGENCY CAKESTAND 3 TIER (≈ 174 157); urmează 23843 (≈ 168 470). **Pe datele brute** primul este `DOT` DOTCOM POSTAGE (≈ 206 245), care nu e produs |
| Luna cu cel mai mare venit | noiembrie 2011 (≈ 1 452 113); cel mai mic din 2011 complet: februarie (≈ 507 783) |
| Decembrie 2011 | ≈ 614 488, dar luna e **incompletă**: datele se opresc la 9 decembrie 2011 |
| Țara | Regatul Unit ≈ 85,1% din venit |
| Clienți pentru 80% din venit | 1 133 din 4 334 clienți identificați (≈ 26,1%) |
| Zile | nu există vânzări sâmbăta; joi are cel mai mare venit |

- [ ] fiecare grafic are titlu-concluzie, axe etichetate și, unde e cazul, o notă despre luna incompletă;
- [ ] am raportat pe date curățate și am spus în raport ce am curățat.

**Interpretare:** Ce concluzie din raport ați da managerului și cu ce rezervă (ce nu putem afirma din aceste date)?

*Răspunsul meu:*

## Pe proiectul vostru

Aplicați același traseu pe datele proiectului: raportul de calitate, curățarea cu decizii motivate și trei grafice cu titlu-concluzie. Dacă proiectul nu are încă date, repetați pe `lab-set1` (`incarca("lab-set1")[0]`).

In [ ]:
# codul pentru proiectul vostru

## Jurnal AI (pe scurt)

| Exercițiu | Cererea principală | Ce am verificat / corectat | Ce am învățat |
|---|---|---|---|
| Exemplele 1-3, 6 |  |  |  |
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |
| Proiect |  |  |  |